# Aralin 18: Pag-secure ng mga AI Agent gamit ang Cryptographic Receipts

## Hands-on Notebook

Pinaglalakaran ng notebook na ito ang apat na pagsasanay:

1. **Pirmahan ang iyong unang resibo** para sa isang tawag sa tool ng agent at i-verify ito.
2. **Mag-tamper sa resibo** at panoorin ang pagkabigo ng beripikasyon.
3. **Bumuo ng tatlong-resibo na kadena** at kumpirmahin ang integridad ng kadena.
4. **I-wrap ang isang Microsoft Agent Framework tool call** upang bawat aksyon ay maglabas ng resibo.

Lahat ng cryptographic primitives ay ini-import mula sa mga mahusay na na-maintain na mga library (`pynacl` para sa Ed25519, `jcs` para sa RFC 8785 canonical JSON, `hashlib` mula sa Python standard library para sa SHA-256). Ang logic ng resibo mismo ay plain Python na maaari mong basahin at baguhin.

Patakbuhin ang mga cells nang sunod-sunod. Bawat seksyon ay maikli at self-contained.


## Setup

I-install ang dalawang dependencies. Pareho silang may mga permissive na lisensya (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Helper Utilities

Ang dalawang helpers na ito ang humahawak ng base64url encoding (na walang padding) at SHA-256 hashing ng iba't ibang bagay. Pinananatili nila ang natitirang bahagi ng notebook na nakatuon sa mismong lohika ng resibo.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Section 1: Lagdaan ang iyong unang resibo

Isipin na ang aming ahente para sa **Contoso Travel** ay kakapanood lang ng mga flight mula Sydney patungong Los Angeles para sa isang customer. Nais naming irekord ang tawag sa tool na ito bilang isang nilagdang resibo upang ang isang tagaseguro sa hinaharap ay makapagpapatunay nito nang hindi kami kailangang pagkatiwalaan.

### Hakbang 1.1: Gumawa ng isang signing key

Sa produksyon, ang signing key ng ahente ay matatagpuan sa hardware security module (HSM), Azure Key Vault, o isang katulad na protektadong imbakan. Para sa leksyong ito, gumagawa kami ng bagong key sa memorya.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Hakbang 1.2: Buuhin ang receipt payload

Ang payload ay naglalaman ng lahat ng nais nating patunayan ng receipt: kung sino ang kumilos, anong kasangkapan, gamit ang anong mga argumento, ano ang bumalik, sa ilalim ng anong patakaran, at kailan. Ina-hash namin ang mga argumento at resulta sa halip na isama ang mga ito nang direktang nakalinya upang hindi mag-leak ng sensitibong nilalaman ang receipt.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Hakbang 1.3: Pirmahan at ipunin ang resibo

Tatlong hakbang:

1. I-canonicalize ang payload gamit ang JCS upang ang dalawang implementasyon na gumagawa ng parehong lohikal na resibo ay makagawa ng byte-identical na bytes.
2. I-hash ang canonical na bytes gamit ang SHA-256.
3. Pirmahan ang hash gamit ang Ed25519 private key.

Ang pirma ay pagkatapos ikinakabit sa orihinal na payload upang makabuo ng panghuling resibo.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### Hakbang 1.4: Suriin ang resibo

Binabaliktad ng beripikasyon ang proseso. Tinatanggal namin ang pirma, muling kino-compute ang canonical hash, at sinusuri ang pirma laban sa pampublikong susi sa resibo.

Ang tagasuri na gumagawa ng beripikasyong ito ay hindi kailangan ng anumang bagay mula sa amin maliban sa mismong resibo. Walang serbisyong dapat tawagan, walang direktoryo ng susi na kailangang itanong, walang pangangailangang pagtitiwala.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

Dapat mong makita ang `Receipt is valid: True`. Ang ahente ay naglabas ng kanyang unang cryptographically signed audit record.


## Section 2: Gambalain ang resibo

Ang buong layunin ng mga resibo ay upang maging halatang may ginawang pagbabago. Patunayan natin ito.

Babaguhin natin ang isang karakter lamang ng resibo at panoorin ang kabiguan ng beripikasyon.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### Ano ang nangyari?

Nang binago namin ang `policy_id`, nagbago ang canonical bytes. Nagbago rin ang SHA-256 hash ng mga bytes na iyon. Ang pirma (na base sa orihinal na hash) ay hindi na tumutugma sa bagong hash. Ang beripikasyon ay tama na nagbalik ng `False`.

Walang paraan upang baguhin ang kahit anong bahagi ng resibo at pa rin itong ma-verify, maliban kung hawak ng attacker ang pribadong susi. Hangga't ang pribadong susi ay nasa key vault at nailathala ang pampublikong susi, imposibleng maitago ang pagbabago.

Subukan mo mismo: baguhin ang `tool_name` o `agent_id` o `timestamp` sa cell sa itaas at muling patakbuhin. Bawat pagbabago ay nagreresulta sa isang invalid na resibo.


## Section 3: Ipagsama-sama ang mga resibo

Isang resibo lang ang nagpoprotekta sa isang aksyon. Karamihan sa mga ahente ay gumagawa ng maraming aksyon. Para maging tamper-evident ang buong pagkakasunod-sunod, ipinagdugtong natin ang bawat resibo sa nakaraang isa sa pamamagitan ng pagsama ng hash ng nakaraang resibo sa payload ng bagong resibo.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Kung may mag-aalis o mag-uusap ng pagkakasunod-sunod ng isang resibo, masisira ang chain sa eksaktong punto na iyon. Mabibigo ang beripikasyon ng anumang huling resibo dahil ang `previous_receipt_hash` nito ay hindi na tumutugma sa aktwal na hash ng nauna nito.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Ngayon sirain ang kadena sa pamamagitan ng pag-manipula sa gitnang resibo at muling pag-verify. Nabibigo ang peke o binagong resibo sa pagsusuri ng lagda, AT nabibigo rin ang susunod na resibo sa pagsusuri ng link ng kadena (dahil ang `previous_receipt_hash` nito ay hindi na tumutugma sa binagong hash ng gitnang resibo).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Sine-verify pa rin ang Resibo 0 (hindi ito na-modify at walang predecessor na pagdedepensahan). Nabigo ang Resibo 1 sa kanyang signature check dahil binago namin ang `tool_args_hash`. Nabigo ang Resibo 2 sa kanyang chain-link check dahil ang `previous_receipt_hash` nito ay kinakalkula laban sa orihinal (na ngayon ay na-modify na) na resibo 1.

Kahit na muling pumirma ang isang attacker sa na-modify na resibo 1 (na hindi nila magagawa nang wala ang private key), ang hindi pagtugma ng chain-link sa resibo 2 ang magpapakita pa rin ng panlilinlang. Upang itago ang pagbabago, kailangang muling pumirma ang attacker sa bawat resibo mula sa punto ng pagbabago pataas, na nangangailangan ng pagmamay-ari ng private key.


## Section 4: Balutin ang pagtawag ng agent tool gamit ang paglagda ng resibo

Sa isang totoong deployment, ayaw mong bawat may-akda ng agent ay alalahanin na tawagan ang `make_receipt`. Gusto mong awtomatiko ang paglagda ng resibo para sa bawat pagtawag ng tool.

Narito ang pinakasimpleng pattern: isang wrapper class na kumukuha ng anumang callable tool function at nagbabalik ng bersyon nito na naglalabas ng resibo. Ito ay umaangkop sa anumang agent framework, kabilang ang Microsoft Agent Framework (`agent_framework.azure`).

Kung wala kang Azure AI Foundry project na naka-setup, ipinapakita pa rin ng lokal na mock sa ibaba ang pattern.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### Pagsasama sa Microsoft Agent Framework

Ang `ReceiptedTool` na wrapper sa itaas ay framework-agnostic. Upang magamit ito sa loob ng isang ahente na ginawa gamit ang Microsoft Agent Framework, irehistro ang naka-wrap na function bilang isang tool. Isang sketch (palitan mo ang mock ng isang totoong Azure AI Foundry tool registration):

```python
# Pseudocode na nagpapakita ng hugis ng integrasyon.
# mula sa agent_framework.azure import AzureAIProjectAgentProvider
# mula sa azure.identity import AzureCliCredential
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = provider.create_agent(
#     instructions="Ikaw ay isang ahente ng Contoso Travel ...",
#     tools=[receipted_lookup],   # ang nakabalot na tool, hindi ang raw na function
# )
# response = agent.run("Maghanap ng mga flight mula Sydney papuntang Los Angeles sa Hunyo.")
#
# # Pagkatapos ng pagtakbo, bawat tawag ng tool na ginawa ng agent ay mayroong signed receipt:
# audit_chain = receipted_lookup.receipts
```

Hindi kailangang malaman ng agent framework ang tungkol sa mga resibo. Ang paglagda ng resibo ay naka-wrap sa paligid ng tool, hindi isinama sa framework. Ganito ka magdagdag ng provenance sa umiiral na code ng ahente nang hindi nire-rewrite ang ahente.


## Pagbabalik-tanaw at hamon sa pagpapalawak

Nagawa mo na:

- Makabuo ng isang Ed25519 key pair.
- Makagawa at mapirmahan ang isang resibo para sa isang agent tool call.
- Mapatunayan ang resibo offline gamit lamang ang public key.
- Mapanirang-buos ang isang resibo at napansin ang kabiguan sa beripikasyon.
- Makabuo ng hash-chained na hanay ng tatlong resibo.
- Mapanirang-buos sa gitna ng kadena at napansin parehong kabiguan ng pirma at kabiguan ng chain-link.
- Nakabalot ng awtomatikong pagpirma ng resibo ang isang tool function.

**Hamon sa pagpapalawak.** Palawakin ang schema ng resibo gamit ang field na `request_id` (isang UUID para sa distributed tracing). I-update ang `make_receipt` para isama ito, at tiyakin na ang mga resibo ay nananatiling mapapatunayan mula simula hanggang dulo. Pagkatapos, baguhin ang field matapos mapirmahan at tiyakin na mapapalala ang verification. Pinipilitan ka nitong unawain kung paano ang bawat byte ng canonical encoding ay nakakatulong sa pirma.

**Mahalagang hangganan.** Pinapatunayan ng mga resibo ang tatlong bagay at tatlong bagay lang: attribution (ang key na ito ang pumirma sa nilalaman), integridad (hindi nagbago ang nilalaman simula nang mapirmahan), at pagkakasunod-sunod (ang resibong ito ay nauna pagkatapos ng ibang resibo). Hindi nito pinapatunayan na tama ang ginawa ng agent, na ang polisiya na tinukoy sa `policy_id` ay totoong nasuri, o na sinunod ng agent ang bawat panuntunan. Ang mga resibo ay pundasyon. Ang pamamahala ay ang sistemang itinatayo mo sa ibabaw nito.

Basahin muli ang README ng aralin na ito nang may dalang pag-unawa sa hangganang iyon. Ang pinaka-karaniwang pagkakamali ng mga koponan sa mga resibo ay ang akalain na "mayroon tayong mga resibo" ay nangangahulugang "namamahalaan na tayo." Hindi ito ganoon. Ginagawa ng mga resibo na masusuri ang pag-uugali ng agent. Hindi nito ginagarantiyahan ang pagiging tama nito.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pagtatanggi**:
Ang dokumentong ito ay isinalin gamit ang serbisyo ng AI translation na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagama't nagsusumikap kami para sa katumpakan, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang maling pagkakaintindi o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
